sft_training-kaggle-multi-experiment.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1yYPCbJvfzZi5Spry7mcwDrfZapTov_hY

# SFT Training — Hafta 3 (Çoklu Deney / Multi-Experiment)

**Proje:** Knowledge Distillation & Reasoning-based Data Augmentation
**Faz:** Student modeli farklı dataset karışım oranlarıyla QLoRA fine-tune

---

## Tasarım

Bu notebook, aşağıdaki **EXPERIMENTS** listesinde tanımladığın TÜM
konfigürasyonları **tek bir çalıştırmada, sırayla** eğitir. Her deney için:

1. Kaggle dosyalarından senin belirlediğin (çakışmasız) karışımla dataset kurulur
2. Temiz (fresh) bir quantized base model + yeni LoRA adapter yüklenir
3. Eğitim yapılır
4. Model, **senin verdiğin isimdeki klasöre** kaydedilir (Drive + yerel + opsiyonel HF push)
5. Kısa bir sanity-check ve bildirim gönderilir
6. Bellek temizlenir, bir sonraki deneye geçilir

Yani örneğin `800a-200b`, `500a-500b`, `3500a-1500b` gibi 3 farklı karışımı
tek run'da art arda eğitip 3 ayrı klasöre kaydettirebilirsin — kodu elle
değiştirip tekrar tekrar çalıştırmana gerek kalmaz.

## 1. Kurulum

In [ ]:
# 1. Hücre - Yeni
import os
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!mkdir -p /content/datasets
!kaggle datasets download -d sewowashere/soru-uzerinde-diyaloglar
!unzip soru-uzerinde-diyaloglar.zip -d /content/datasets/
print("✓ Datasetler /content/datasets altına indirildi.")

In [ ]:
# GPU kontrolü
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Commented out IPython magic to ensure Python compatibility.
%%capture
# # Qwen3.5 desteği için transformers'in main'i şart
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q -U trl peft accelerate bitsandbytes datasets huggingface_hub
!pip install -q kagglehub[pandas-datasets]

## 2. Genel Konfigürasyon (tüm deneylerde ORTAK)

Bu hücredeki değerler tüm deneyler için sabittir — fair comparison için
hyperparam'lar deneyler arasında değişmiyor. Deney bazlı ayarlar (karışım,
klasör ismi) bir sonraki hücrede (`EXPERIMENTS`).

In [ ]:
STUDENT_MODEL = "Qwen/Qwen3.5-4B"  # <-- alternatif: "Qwen/Qwen3-8B"

In [ ]:
# ── Hyperparameters (tüm deneylerde aynı) ────────────────────────────────
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8        # effective batch = 2 × 8 = 16
MAX_SEQ_LENGTH = 2048       # multi-turn için yeterli, scale için fazla
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01

In [ ]:
# LoRA config
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

In [ ]:
# Training mode: assistant turn'lerinde loss
# True → student sadece teacher'ın "asistan cevaplarını" öğrenir, user mesajları masklenir
ASSISTANT_ONLY_LOSS = False  # Qwen3.5 chat template ile uyumsuz, kapatıldı

In [ ]:
# Logging / saving
LOGGING_STEPS = 5
SAVE_STEPS = 100
EVAL_STEPS = 50

In [ ]:
# ── Çıktı klasörlerinin taban dizinleri ──────────────────────────────────
# Her deney bu tabanların altına KENDİ ismiyle bir alt klasör açar
# (örn. .../sft_outputs_retrain/800a-200b/, .../cosmos/500a-500b/)
from pathlib import Path
OUTPUT_DIR_BASE = Path("/content/drive/MyDrive/colab/sft_outputs_retrain")
SAVE_DIR_BASE = Path("/content/drive/MyDrive/cosmos")

In [ ]:
# HF push açık mı? (her deney kendi adapter repo'sunu oluşturur)
PUSH_TO_HF = True
HF_USERNAME = "AliMertTemizsoy"  # <-- DUZENLE

In [ ]:
import os
import json
import gc
import random
import torch
import numpy as np
from datetime import datetime
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [ ]:
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# HF token (model gating + push için)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

In [ ]:
print(f"HF token: {'OK' if HF_TOKEN else 'YOK!'}")

## 3. Deney Listesi — EXPERIMENTS

**Burayı düzenle.** Her sözlük (`dict`) bir deneyi temsil eder:

- `"name"`     → çıktı klasörünün ve HF repo'sunun ismi (senin verdiğin isim)
- `"mix_mode"` → `"count"` (elle örnek sayısı) veya `"ratio"` (oran + `total_dataset_size`)
- `"total_dataset_size"` → sadece `mix_mode="ratio"` iken kullanılır
- `"mix_config"` → `(dosya_adı, miktar/oran)` çiftlerinin listesi

**Karışım mantığı** (önceki bölümle aynı, çakışmasız & seed'siz):
Tüm Kaggle dosyaları aynı GSM8K sorularının farklı çözümleridir ve satır
sırası dosyalar arasında aynıdır. Global bir "soru index"i 0'dan başlar;
`mix_config`'teki her satır bu index'ten ardışık, çakışmayan bir dilim alır.
Örn. `("kritik-nokta.json", 800), ("tuzaklar-1000.json", 200)` dersen:
ilk 800 soru kritik-nokta'dan, sonraki 200 soru (800-1000 aralığı)
tuzaklar'dan gelir — hiçbir soru iki dosyadan birden alınmaz.

In [ ]:
EXPERIMENTS = [
    {
        "name": "800a-200b",
        "mix_mode": "count",
        "mix_config": [
            ("gsm8k_gemini_2_5_sonuclari-kritik-nokta.json", 800),
            ("gsm8k_gemini_2_5_sonuclari-tuzaklar-1000.json", 200),
        ],
    },
    {
        "name": "500a-500b",
        "mix_mode": "count",
        "mix_config": [
            ("gsm8k_gemini_2_5_sonuclari-kritik-nokta.json", 500),
            ("gsm8k_gemini_2_5_sonuclari-tuzaklar-1000.json", 500),
        ],
    },
    {
        "name": "3500a-1500b",
        "mix_mode": "count",
        "mix_config": [
            ("gsm8k_gemini_2_5_sonuclari-kritik-nokta-2k.json", 3500),
            ("gsm8k_gemini_2_5_sonuclari-tuzaklar-2000.json", 1500),
        ],
    },
    # "ratio" modu örneği (TOTAL_DATASET_SIZE'a göre otomatik hesaplanır):
    # {
    #     "name": "acc-weighted-1000",
    #     "mix_mode": "ratio",
    #     "total_dataset_size": 1000,
    #     "mix_config": [
    #         ("gsm8k_gemini_2_5_sonuclari-kritik-nokta.json", 0.53),
    #         ("gsm8k_gemini_2_5_sonuclari-tuzaklar-1000.json", 0.31),
    #         ("gsm8k_gemini_2_5_sonuclari-neler-bilmek-gerekir (tam).json", 0.16),
    #     ],
    # },
]

In [ ]:
print(f"{len(EXPERIMENTS)} deney tanımlandı:")
for exp in EXPERIMENTS:
    print(f"  - {exp['name']}")

## 4. Kaggle Dosyalarını Yükle (cache) & Karışım Fonksiyonu

Tüm deneylerde geçen Kaggle dosyalarını **bir kere** indirip belleğe alıyoruz
(aynı dosya birden fazla deneyde geçse bile tekrar indirmez).

In [ ]:
KAGGLE_DATASET = "sewowashere/soru-uzerinde-diyaloglar"

In [ ]:
_dataframes_cache = {}

In [ ]:
def get_dataframe(file_name):
    """Kaggle dosyasını cache'ten döndürür, yoksa indirir."""
    if file_name not in _dataframes_cache:
        print(f"Yükleniyor: {file_name}")
        df = kagglehub.load_dataset(
            KaggleDatasetAdapter.PANDAS,
            KAGGLE_DATASET,
            file_name,
        )
        df = df.reset_index(drop=True)
        print(f"  {len(df)} kayıt bulundu")
        _dataframes_cache[file_name] = df
    return _dataframes_cache[file_name]

In [ ]:
def build_mixed_dataset(mix_config, mix_mode="count", total_dataset_size=None):
    """
    mix_config'teki (dosya_adi, miktar/oran) çiftlerinden, ARDIŞIK ve
    ÇAKIŞMASIZ global index dilimleme yaparak birleşik bir dataset kurar.
    Seed kullanılmaz — tamamen deterministiktir.
    Döner: (Dataset, mix_summary listesi)
    """
    all_examples = []
    mix_summary = []
    global_offset = 0

    for file_name, amount in mix_config:
        df = get_dataframe(file_name)

        if mix_mode == "ratio":
            if not total_dataset_size:
                raise ValueError("mix_mode='ratio' için total_dataset_size gerekli.")
            count = round(amount * total_dataset_size)
        elif mix_mode == "count":
            count = int(amount)
        else:
            raise ValueError(f"Bilinmeyen mix_mode: {mix_mode!r} ('count' ya da 'ratio' olmalı)")

        start, end = global_offset, global_offset + count

        if end > len(df):
            raise ValueError(
                f"'{file_name}' dosyası [{start}:{end}) aralığını karşılayamıyor "
                f"(dosyada sadece {len(df)} kayıt var). mix_config'teki miktarları "
                f"ya da sırasını gözden geçir."
            )

        subset = df.iloc[start:end]
        for _, row in subset.iterrows():
            messages = [
                {"role": "user", "content": str(row["question"])},
                {"role": "assistant", "content": str(row["model_output"])},
            ]
            all_examples.append({"messages": messages})

        mix_summary.append((file_name, start, end, count))
        global_offset = end

    return Dataset.from_list(all_examples), mix_summary

In [ ]:
def print_mix_summary(mix_summary):
    print("── Karışım özeti ──────────────────────────────────────────────")
    total = 0
    for file_name, start, end, count in mix_summary:
        print(f"  {file_name:<55} [{start:>5}:{end:<5})  {count:>5} örnek")
        total += count
    print(f"  {'TOPLAM':<55} {'':<14} {total:>5} örnek")
    print("──────────────────────────────────────────────────────────────")

## 5. Tokenizer (tüm deneylerde ortak, bir kere yüklenir)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    STUDENT_MODEL, token=HF_TOKEN, trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # SFT için right padding daha iyi

In [ ]:
print(f"Chat template var mı: {tokenizer.chat_template is not None}")

In [ ]:
compute_dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",  # Qwen3 için tüm projection'lar
)

In [ ]:
def load_fresh_model():
    """Her deney için TEMİZ bir quantized base model + yeni LoRA adapter yükler.
    Önceki deneyin ağırlıkları/adapter'ı asla bir sonrakine sızmaz."""
    model = AutoModelForCausalLM.from_pretrained(
        STUDENT_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        token=HF_TOKEN,
        trust_remote_code=True,
        torch_dtype=compute_dtype,
    )
    model.config.use_cache = False
    model.config.pretraining_tp = 1
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(model, lora_config)
    return model

## 6. Bildirim & Sanity-Check Yardımcı Fonksiyonları

In [ ]:
import requests

In [ ]:
def send_push_alert(title, message):
    api_key = "3193d93e-6601-4ea7-995f-f693a6a1139d"
    url = f"https://push.techulus.com/api/v1/notify/{api_key}"
    payload = {"title": title, "body": message}
    headers = {"Content-Type": "application/json"}
    try:
        response = requests.post(url, headers=headers, data=json.dumps(payload), timeout=10)
        if response.status_code == 200:
            print("Bildirim başarıyla gönderildi! ✅")
        else:
            print(f"Hata Kodu: {response.status_code} | Yanıt: {response.text}")
    except Exception as e:
        print(f"Bir bağlantı hatası oluştu: {e}")

In [ ]:
TEST_QUESTIONS = [
    "Ali'nin 5 elması, Ayşe'nin ise Ali'nin 3 katı kadar elması var. Toplam kaç elma vardır?",
    "Bir kitap 25 TL. Mert 4 kitap aldıysa ve 10 TL indirim aldıysa kaç TL ödedi?",
]

In [ ]:
def run_sanity_check(model, tokenizer):
    model.eval()
    for q in TEST_QUESTIONS:
        msgs = [
            {"role": "user", "content": f"Aşağıdaki matematik sorusunu adım adım çözer misin? Son cevabı '#### [Sayı]' formatında belirt.\n\nSoru: {q}"}
        ]
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=400,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        print(f"Q: {q}")
        print(f"A: {text[:500]}...\n{'-'*70}")

## 7. Ana Döngü — Tüm Deneyleri Sırayla Çalıştır

Her deney için: dataset kur → temiz model yükle → eğit → kaydet (Drive + yerel
+ opsiyonel HF push) → sanity-check → bildirim → belleği temizle → sıradaki deney.

In [ ]:
all_results = []

In [ ]:
for exp_idx, exp in enumerate(EXPERIMENTS, start=1):
    exp_name = exp["name"]
    mix_mode = exp.get("mix_mode", "count")
    total_dataset_size = exp.get("total_dataset_size")
    mix_config = exp["mix_config"]

    print("\n" + "=" * 80)
    print(f"DENEY {exp_idx}/{len(EXPERIMENTS)}: {exp_name}")
    print("=" * 80)

    # ── 1) Dataset kur ────────────────────────────────────────────────────
    train_ds, mix_summary = build_mixed_dataset(mix_config, mix_mode, total_dataset_size)
    print_mix_summary(mix_summary)

    split = train_ds.train_test_split(test_size=0.05, seed=SEED)
    train_ds, eval_ds = split["train"], split["test"]
    print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")

    # ── 2) Temiz model yükle ────────────────────────────────────────────
    model = load_fresh_model()
    print(f"Model yüklendi ({exp_name}). VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

    # ── 3) Trainer & eğitim ─────────────────────────────────────────────
    exp_output_dir = OUTPUT_DIR_BASE / exp_name
    exp_output_dir.mkdir(parents=True, exist_ok=True)

    sft_config = SFTConfig(
        output_dir=str(exp_output_dir),
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_steps=50,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type="cosine",
        optim="paged_adamw_8bit",
        bf16=(compute_dtype == torch.bfloat16),
        fp16=False,
        gradient_checkpointing=False,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        max_length=MAX_SEQ_LENGTH,
        packing=False,
        assistant_only_loss=ASSISTANT_ONLY_LOSS,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        save_total_limit=2,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        report_to="none",
        seed=SEED,
        dataloader_num_workers=2,
        remove_unused_columns=True,
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        processing_class=tokenizer,
    )

    train_result = trainer.train()
    final_loss = float(train_result.training_loss)
    print(f"[{exp_name}] Eğitim tamamlandı! Final loss: {final_loss:.4f}")

    # ── 4) Kaydet: Drive'daki SENİN VERDİĞİN isimdeki klasöre ──────────
    exp_save_dir = SAVE_DIR_BASE / exp_name
    trainer.save_model(str(exp_save_dir))
    tokenizer.save_pretrained(str(exp_save_dir))

    with open(exp_save_dir / "train_metrics.json", "w") as f:
        json.dump({
            "experiment_name": exp_name,
            "student": STUDENT_MODEL,
            "mix_mode": mix_mode,
            "mix_config": mix_config,
            "mix_summary": mix_summary,
            "final_loss": final_loss,
            "epochs": NUM_EPOCHS,
            "train_size": len(train_ds),
            "eval_size": len(eval_ds),
            "timestamp": datetime.now().isoformat(),
        }, f, indent=2, ensure_ascii=False)

    print(f"[{exp_name}] Model kaydedildi: {exp_save_dir}")

    # ── 5) Sanity check ──────────────────────────────────────────────────
    run_sanity_check(model, tokenizer)

    # ── 6) Opsiyonel: HF Hub push ──────────────────────────────────────
    if PUSH_TO_HF:
        hf_repo = f"{HF_USERNAME}/qwen3_5-4b-{exp_name}"
        model.push_to_hub(hf_repo, token=HF_TOKEN, private=True)
        tokenizer.push_to_hub(hf_repo, token=HF_TOKEN, private=True)
        print(f"[{exp_name}] Push edildi: https://huggingface.co/{hf_repo}")

    # ── 7) Bildirim ───────────────────────────────────────────────────────
    send_push_alert(
        f"Deney bitti: {exp_name}",
        f"({exp_idx}/{len(EXPERIMENTS)}) final loss: {final_loss:.4f} — {exp_save_dir}",
    )

    all_results.append({
        "experiment_name": exp_name,
        "final_loss": final_loss,
        "train_size": len(train_ds),
        "eval_size": len(eval_ds),
        "save_dir": str(exp_save_dir),
    })

    # ── 8) Belleği temizle, sıradaki deneye hazırlan ────────────────────
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
print("\n" + "=" * 80)
print("TÜM DENEYLER TAMAMLANDI")
print("=" * 80)
for r in all_results:
    print(f"  {r['experiment_name']:<25} final_loss={r['final_loss']:.4f}  "
          f"train={r['train_size']}  eval={r['eval_size']}  ->  {r['save_dir']}")

## 8. Genel Özet Kaydet & Toplu Bildirim

In [ ]:
summary_path = SAVE_DIR_BASE / "all_experiments_summary.json"
with open(summary_path, "w") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

In [ ]:
print(f"Özet kaydedildi: {summary_path}")

In [ ]:
send_push_alert(
    "Tüm deneyler bitti",
    f"{len(all_results)} deney tamamlandı. git bilgisayarı kontrol et.",
)

## Sonraki Adım

Her deney kendi klasörüne (`SAVE_DIR_BASE/<deney_ismi>/`) ve (açıksa) kendi HF
repo'suna (`qwen3_5-4b-<deney_ismi>`) kaydedildi. Hafta 4 evaluation
notebook'unda `all_experiments_summary.json`'daki listeyi kullanarak tüm
adapter'ları + raw baseline'ı GSM8K-TR **test split** üzerinde karşılaştır.